# Homework 3: Diffusion Models
To help you get started with the homework, we have provided a template code that covers the essential components of the diffusion model.
You are required to modify and expand upon this template to implement the U-Net model and complete the diffusion process for MNIST image generation.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")  # Use CUDA (GPU) if available
else:
    device = torch.device("cpu")  # Use CPU if CUDA is not available
print(device)

cuda


## Hyperparameter setting

In [3]:
batch_size = 128
epochs = 15
lr = 1e-3
image_size = 28
timesteps = 1000
beta_start = 0.0005
beta_end = 0.01

## MNIST dataset loading
- setup for loading the MNIST dataset, applying necessary preprocessing steps, and converting the images to the format required for the diffusion process
- Ensure that you check the data loading functions and modify them if you choose to experiment with additional preprocessing techniques.

In [4]:
# Loading MNIST dataset from pytorch
transform = transforms.Compose([
    # transformed from [0,255] to [0,1]
    transforms.ToTensor(),
    # transforms your data in a range [-1, 1]
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 9.91M/9.91M [00:00<00:00, 16.2MB/s]


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 28.9k/28.9k [00:00<00:00, 498kB/s]


Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 1.65M/1.65M [00:00<00:00, 4.57MB/s]


Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 4.54k/4.54k [00:00<00:00, 2.95MB/s]

Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw



## Set β, α and  ̄α

In [5]:
def linear_beta_schedule(timesteps):
    return torch.linspace(beta_start, beta_end, timesteps)

betas = linear_beta_schedule(timesteps).to(device)
alphas = (1 - betas).to(device)
alphas_bar = torch.cumprod(alphas.cpu(), axis=0).to(device)

sqrt_alphas_bar = torch.sqrt(alphas_bar).to(device)
sqrt_one_minus_alphas_bar = torch.sqrt(1 - alphas_bar).to(device)

sigmas = torch.zeros(timesteps).to(device)
for t in range(1, timesteps):
    sigmas[t] = torch.sqrt(betas[t] * (1 - alphas_bar[t-1]) / (1 - alphas_bar[t]))

## Forward process
- This section defines the forward process, where random Gaussian noise is added to the input images over several timesteps. Each step gradually transforms a clean image into a noisy version, simulating the process of corruption.
- This forward process is crucial for training the model, as the reverse process will attempt to undo these transformations.

In [6]:
def forward_diffusion(x_0, t, noise):
    ## broadcast to (batch_size, 1, 1, 1)
    sqrt_alphas_bar_t = sqrt_alphas_bar[t][:, None, None, None].to(device)
    sqrt_one_minus_alphas_bar_t = sqrt_one_minus_alphas_bar[t][:, None, None, None].to(device)
    return sqrt_alphas_bar_t * x_0 + sqrt_one_minus_alphas_bar_t* noise

## UNet model definition
- You are expected to extend this structure by designing the appropriate downsampling, bottleneck, and upsampling layers.
- The inputs to your model should include the noisy image, timestep, and class label, and the output should be the predicted noise, which will be subtracted from the noisy image in the reverse process.
- Make sure to experiment with different architectures (e.g., kernel sizes, depths, number of filters) to improve the model’s performance.

In [25]:
def resize_to_match(x, y):
    return F.interpolate(x, size=(y.size(2), y.size(3)), mode='bilinear', align_corners=False)

In [26]:
class UNet(nn.Module):
    def __init__(self, num_classes, num_channels=1, base_channels=64):
        super(UNet, self).__init__()

        # Embedding layers for timestep and class label
        self.time_embed = nn.Linear(1, base_channels)
        self.class_embed = nn.Embedding(num_classes, base_channels)

        # Downsampling layers
        self.down1 = self.conv_block(num_channels + 2 * base_channels, base_channels)
        self.down2 = self.conv_block(base_channels, base_channels * 2)
        self.down3 = self.conv_block(base_channels * 2, base_channels * 4)

        # Bottleneck
        self.bottleneck = self.conv_block(base_channels * 4, base_channels * 8)

        # Upsampling layers
        self.up3 = self.up_block(base_channels * 8, base_channels * 4)
        self.up2 = self.up_block(base_channels * 4, base_channels * 2)
        self.up1 = self.up_block(base_channels * 2, base_channels)

        # Final output
        self.final_conv = nn.Conv2d(base_channels, num_channels, kernel_size=1)

    def conv_block(self, in_channels, out_channels):
        """A convolutional block with two Conv2D layers followed by ReLU."""
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def up_block(self, in_channels, out_channels):
        """An upsampling block using ConvTranspose2D."""
        return nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_noisy, timesteps, target_class):
      # Embed timestep and class
      time_emb = self.time_embed(timesteps.float().unsqueeze(-1)).unsqueeze(-1).unsqueeze(-1)  # (B, C, 1, 1)
      class_emb = self.class_embed(target_class.long()).unsqueeze(-1).unsqueeze(-1)  # (B, C, 1, 1)

      # Ensure time_emb and class_emb match the channels of x_noisy
      time_emb = time_emb.expand(-1, -1, x_noisy.size(2), x_noisy.size(3))  # Expand to match height/width
      class_emb = class_emb.expand(-1, -1, x_noisy.size(2), x_noisy.size(3))  # Expand to match height/width

      # Concatenate embeddings with input
      x = torch.cat([x_noisy, time_emb, class_emb], dim=1)  # Concatenate along channel dimension

      # Downsampling path
      d1 = self.down1(x)
      d2 = self.down2(F.max_pool2d(d1, 2))
      d3 = self.down3(F.max_pool2d(d2, 2))

      # Bottleneck
      b = self.bottleneck(F.max_pool2d(d3, 2))

      # Upsampling path
      u3 = self.up3(b)
      u3 = resize_to_match(u3, d3)  # Ensure u3 matches d3 dimensions

      u2 = self.up2(u3 + d3)
      u2 = resize_to_match(u2, d2)  # Ensure u2 matches d2 dimensions

      u1 = self.up1(u2 + d2)
      u1 = resize_to_match(u1, d1)  # Ensure u1 matches d1 dimensions

      # Final output
      out = self.final_conv(u1 + d1)
      return out

## Backward process
- This section is to implement the reverse diffusion process, where the noisy images are progressively denoised using the predictions from your U-Net model.
- You need through iterating from the last timestep (T) to the first, gradually reconstructing the image.

In [27]:
def backward_process(model, noisy_image, timesteps, target_class, scheduler, num_steps):

    device = noisy_image.device
    batch_size, channels, height, width = noisy_image.size()

    # Start with the noisy image
    current_image = noisy_image.clone()

    for step in reversed(range(1, num_steps + 1)):
        t = torch.full((batch_size,), step, device=device, dtype=torch.long)

        # Predict the noise using the model
        predicted_noise = model(current_image, t, target_class)

        # Scheduler provides step-specific variance and noise scaling factors
        alpha_t, alpha_t_minus_1, sigma_t = scheduler(step)

        # Remove predicted noise and add sampled noise
        current_image = (
            (1 / torch.sqrt(alpha_t_minus_1)) *
            (current_image - (1 - alpha_t) / torch.sqrt(1 - alpha_t) * predicted_noise)
        )

        if step > 1:
            noise = torch.randn_like(current_image)  # Add noise for stochasticity
            current_image += sigma_t * noise

    return current_image

In [28]:
class LinearBetaScheduler:
    def __init__(self, timesteps, beta_start, beta_end):
        self.timesteps = timesteps
        self.beta = torch.linspace(beta_start, beta_end, timesteps)
        self.alpha = 1.0 - self.beta
        self.alpha_cumprod = torch.cumprod(self.alpha, dim=0)

    def __call__(self, t):
        t_index = t - 1  # t is 1-indexed, array is 0-indexed
        alpha_t = self.alpha_cumprod[t_index]
        alpha_t_minus_1 = (
            self.alpha_cumprod[t_index - 1] if t_index > 0 else torch.tensor(1.0)
        )
        sigma_t = torch.sqrt(self.beta[t_index])
        return alpha_t, alpha_t_minus_1, sigma_t

In [29]:
scheduler = LinearBetaScheduler(timesteps=timesteps, beta_start=beta_start, beta_end=beta_end)

## Visualization
- This section you need to visualize the generate sample image.
- In order to get the image, you will need to call Backward process function.
- You will modify this section to generate the 3x10 image grids at each epoch, as well as create the final GIF that demonstrates the denoising process through iterating from the last timestep (T) to the first, gradually reconstructing the image.

In [30]:
from google.colab import drive
drive.mount('/content/drive')
save_dir = "/content/drive/My Drive/HW3_113064525_徐綉惠/generated_result"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
import torchvision.utils as vutils
import os

def sample_image_grid(model, timesteps, betas, alphas, alphas_bar, sigmas, device, epoch, save_dir):

    os.makedirs(save_dir, exist_ok=True)

    # Parameters
    num_classes = 10  # Digits 0 to 9
    rows = 3          # 3 different noise samples per class
    cols = num_classes
    num_samples = rows * cols  # Total samples (30 in this case)

    # Initialize noisy images for each class (3 rows per class)
    x_t = torch.randn(num_samples, 1, 28, 28).to(device)  # Noisy images at timestep T=0

    # Repeat labels for each class
    labels = torch.tensor([i for _ in range(rows) for i in range(cols)], dtype=torch.long, device=device)  # 3 times each class

    # Loop for reverse diffusion
    for t in range(timesteps - 1, -1, -1):
        t_tensor = torch.tensor([t], device=device).repeat(num_samples)  # Current timestep for all samples

        # Predict noise using the model
        predicted_noise = model(x_t, t_tensor / timesteps, labels)

        # Ensure predicted_noise has the same shape as x_t
        assert predicted_noise.shape == x_t.shape, f"Shape mismatch: {predicted_noise.shape} vs {x_t.shape}"

        # Calculate parameters for denoising
        sqrt_alpha_t = torch.sqrt(alphas[t])
        sqrt_one_minus_alpha_bar_t = torch.sqrt(1 - alphas_bar[t])
        beta_t = betas[t]
        sigma_t = sigmas[t]

        # Add noise for all but the last step
        if t > 0:
            z = torch.randn_like(x_t).to(device)
        else:
            z = torch.zeros_like(x_t).to(device)

        # Perform reverse diffusion
        x_t = (1 / sqrt_alpha_t) * (x_t - (beta_t / sqrt_one_minus_alpha_bar_t) * predicted_noise) + sigma_t * z

    # Reshape and save the grid (3 rows x 10 columns)
    grid = vutils.make_grid(x_t, nrow=cols, normalize=True, value_range=(-1, 1), pad_value=1)
    file_path = os.path.join(save_dir, f"epoch_{epoch}.png")
    vutils.save_image(grid, file_path)

In [32]:
import imageio

def sample_generating_process(model, timesteps, betas, alphas, alphas_bar, sigmas, device, save_dir, target_class=4):

    os.makedirs(save_dir, exist_ok=True)

    # Initialize noisy image
    x_t = torch.randn(1, 1, 28, 28).to(device)  # Starting from pure noise
    class_label = torch.tensor([target_class], dtype=torch.long, device=device)  # Set class to target_class

    images = []  # To store images at each timestep for the GIF

    # Perform reverse diffusion
    for t in range(timesteps - 1, -1, -1):
        t_tensor = torch.tensor([t], device=device)  # Current timestep

        # Predict noise using the model
        predicted_noise = model(x_t, t_tensor / timesteps, class_label)  # Normalize timestep

        # Compute reverse diffusion update
        sqrt_alpha_t = torch.sqrt(alphas[t])
        sqrt_one_minus_alpha_bar_t = torch.sqrt(1 - alphas_bar[t])
        beta_t = betas[t]
        sigma_t = sigmas[t]

        # Add noise unless it's the last step
        z = torch.randn_like(x_t).to(device) if t > 0 else torch.zeros_like(x_t).to(device)

        x_t = (1 / sqrt_alpha_t) * (x_t - (beta_t / sqrt_one_minus_alpha_bar_t) * predicted_noise) + sigma_t * z

        # Store the intermediate denoising steps as images
        grid = vutils.make_grid(x_t, nrow=1, normalize=True, value_range=(-1, 1))
        img = grid.permute(1, 2, 0).cpu().numpy() * 255  # Convert to numpy for GIF
        images.append(img.astype(np.uint8))

    # Save the images as a GIF
    gif_path = os.path.join(save_dir, f"denoising_process.gif")
    imageio.mimsave(gif_path, images, fps=10)

## Training process
- The provided training loop handles multiple epochs of training. For each epoch, the model processes noisy images and learns to predict the added noise.

- You are required to modify the loss function and optimization strategy, if necessary, to enhance the model's learning.
- By the end of the training, the model should be able to produce high-quality images from noise.


In [33]:
mse_loss = nn.MSELoss()

In [34]:
def train_model(model, train_loader, optimizer, epochs, timesteps, betas, alphas, alphas_bar, sigmas, device):
    model.train()
    for epoch in range(1, epochs + 1):  # Start epoch from 1
        epoch_loss = 0
        num_batches = 0  # Track the number of batches in each epoch

        for batch_idx, (data, target) in enumerate(train_loader):
            data = data.to(device)
            target = target.to(device).long()

            # Generate random timesteps with proper dtype
            t = torch.randint(0, timesteps, (data.shape[0],), device=device, dtype=torch.long)

            # Add noise to the data
            noise = torch.randn_like(data).to(device)
            x_noisy = forward_diffusion(data, t, noise)  # Ensure forward_diffusion handles t correctly

            # Predict noise using the model
            noise_pred = model(x_noisy, t / timesteps, target)

            # Compute loss and backpropagate
            loss = mse_loss(noise_pred, noise)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            num_batches += 1  # Increment batch counter

        # Calculate the average loss for the epoch
        average_loss = epoch_loss / num_batches
        print(f"Epoch [{epoch}/{epochs}] Average Loss: {average_loss:.6f}")

        # Sample images and save after each epoch
        with torch.no_grad():
            sample_image_grid(model, timesteps, betas, alphas, alphas_bar, sigmas, device, epoch, save_dir)

        # Final epoch: run the generation process
        if epoch == epochs:
            with torch.no_grad():
                sample_generating_process(model, timesteps, betas, alphas, alphas_bar, sigmas, device, save_dir)

## Start training

In [35]:
model = UNet(num_classes=10).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)

## If you don't want to train the model again, please skip this section.

In [36]:
train_model(model, train_loader, optimizer, epochs, timesteps, betas, alphas, alphas_bar, sigmas, device)

Epoch [1/15] Average Loss: 0.225455
Epoch [2/15] Average Loss: 0.066301
Epoch [3/15] Average Loss: 0.050320
Epoch [4/15] Average Loss: 0.045983
Epoch [5/15] Average Loss: 0.054290
Epoch [6/15] Average Loss: 0.042577
Epoch [7/15] Average Loss: 0.040019
Epoch [8/15] Average Loss: 0.038671
Epoch [9/15] Average Loss: 0.036149
Epoch [10/15] Average Loss: 0.035784
Epoch [11/15] Average Loss: 0.035494
Epoch [12/15] Average Loss: 0.035256
Epoch [13/15] Average Loss: 0.034382
Epoch [14/15] Average Loss: 0.033429
Epoch [15/15] Average Loss: 0.244884


## Save model

In [19]:
def save_model_as_npy(model, filepath):
    model_weights = model.state_dict()
    model_weights_dict = {k: v.cpu().numpy() for k, v in model_weights.items()}
    np.save(filepath, model_weights_dict)

In [ ]:
save_model_as_npy(model, 'best_model_weight.npy')